# 모두마켓 월간 로그 — 도구 선택 보고서

## 1. 데이터 개요
- 파일: web_logs.csv
- 크기: 32.3 MB, 500,000 행 (메모리 적재 시 약 101.6 MB)
- 주요 컬럼: ts / user_id / page / device / status_code / response_ms / bytes_sent

## 2. 측정 결과
| 방식 | 소요 시간 | 메모리 피크 |
|---|---|---|
| pandas + dtype | 0.84초 | +39 MB |
| pandas chunked | 0.60초 | +1 MB |
| Polars lazy    | 0.30초 | +59 MB |

## 3. 분석 결과 요약
- 페이지별 평균 응답 시간 최고: cart (160.9ms) — 페이지별 차이는 미미(159~161ms)
- 디바이스 점유율: mobile 70.0%, desktop 25.0%, tablet 5.0%
- 에러(status>=400) 1위 페이지: home (43,006건) → list, detail 순

## 4. 도구 선택 정당화 (한 단락)
이 분석에는 Polars lazy를 선택했습니다. 이유는 
(1) 측정 결과
소요 시간이 0.30초로 세 방식 중 가장 빨랐고,
필터+집계(에러 카운트)에서 predicate pushdown 덕을 봤기 때문입니다.
(2) 매일 자동 실행되는 파이프라인이라면 재현성과 속도가 중요한데
scan_csv → filter → agg → collect 단일 파이프라인으로 깔끔하게 표현됩니다.
(3) 다만 메모리가 극도로 부족한 환경이라면 메모리 증가가 +1MB에 그친
청크 처리가 유일한 대안이 될 수 있어, 상황에 따라 병행을 고려합니다.

## 5. 다음 단계 제안
- 데이터 크기가 1억 행 이상으로 늘어나면 Polars lazy + Parquet 저장으로 전환 고려.
- 시각화·검증은 다음 시간에 Seaborn/Plotly를 활용해 진행.